# Lab 08-01 — Entity/relation graph from passages (the GraphRAG index)

**Track 08 · GraphRAG** — how we index the *structure* of a corpus instead of just its chunks.

Plain RAG embeds chunks and retrieves by vector similarity; GraphRAG changes what gets indexed. Before storing anything, it asks a local LLM to read each passage and pull out the **entities** and the **relations between them** as `(head, relation, tail)` triples, then folds those triples into one graph. The pipeline, drawn inline:

`passages (rag-mini-wikipedia parquet) → ChatOllama json_object triple extraction (qwen2.5-coder:7b @ localhost:11434) → (head, relation, tail) triples → networkx entity graph → stats / hubs / example triples`

This notebook is **self-contained**: it imports `langchain-ollama`, `pandas`, and `networkx` directly — no repo component library. Every block of the pipeline is built right here: the local-LLM wrapper (a `ChatOllama` chat model with markdown-code-fence stripping and JSON retries), the triple extractor, and the graph builder all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The graph is the artifact everything else in this track builds on: lab 03 detects communities over it and lab 04 queries it locally and globally. Nodes are entities, edges carry the relation phrases that link them, and each node remembers which passages mention it — a lossy but *structured* summary of the corpus, and the bridge structure dense retrieval cannot see.


## Setup

Two prerequisites must hold before this notebook will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM every triple-extraction call goes to (talked to through `langchain-ollama`). Fully local: no API key, no quota. If the server is not up, every extraction fails and the graph never materializes.
- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet`, already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-ollama`, `pandas`, and `networkx`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no sys.path trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   langchain-ollama  -> ChatOllama, the local Ollama chat backend
#   pandas            -> read the rag-mini-wikipedia parquet
#   networkx          -> the entity graph object
%pip install -q langchain-ollama pandas networkx


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

# pandas + networkx + langchain-ollama — the only libraries this notebook
# needs. Nothing is imported from the repo's src/ component library.
import networkx as nx
import pandas as pd
from langchain_ollama import ChatOllama

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 25` takes the **deterministic head** of rag-mini-wikipedia — and because every passage costs exactly one LLM extraction call, this number *is* the runtime knob: 25 passages, 25 local calls, no sampling variance between runs. `TOP_K` and `N_SAMPLE_RELATIONS` only shape the demo output (how many hubs to list, how many example triples to show), so they stay small.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 25  # deterministic head; each passage costs one LLM extraction call
TOP_K = 8  # how many highest-degree entities to print
N_SAMPLE_RELATIONS = 8  # how many example (head, relation, tail) triples to show


## 2. Load — first N passages of rag-mini-wikipedia

`passages.parquet` is a plain table with a `passage` column; `head(n)` keeps the first `n` rows so every run works on the same corpus slice. Each passage is loaded **whole**: there is no chunking here, because the LLM extractor consumes the full text at once. That is the deliberate contrast with every earlier track — the unit of analysis has moved from a chunk to a passage.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — extract triples per passage and fold them into one graph

This is the GraphRAG index step, in two halves, both built inline:

- **Extract** — the repo's `OllamaLLM` adapter is replaced by a plain `_OllamaLLM` class right here: a `ChatOllama(model="qwen2.5-coder:7b", temperature=0.0)` chat model whose `json_object` strips a surrounding markdown code fence (local models love wrapping JSON in ```json fences) and re-prompts up to two times when the JSON does not parse. `extract_triples` sends each passage the same prompt and schema the lab uses and tolerates a `{"triples": [...]}` wrapper, an alias key, or a bare array.
- **Fold** — `build_entity_graph` turns every passage's triples into one networkx graph: nodes are entities, edges carry the extracted relation phrases in `relations`, each edge's `weight` is the number of distinct relation phrases, and every node records the passage indices that mention it. `graph_stats`, `top_entities`, and `sample_relations` read that structure back out.

One local LLM call per passage (`N_PASSAGES = 25`), with a `progress` callback counting `extracted done/total passages` on a single line — the same contract the lab's shared graph tool wraps.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — extract triples per passage and fold them into one graph
# --------------------------------------------------------------------------
# Inline replacement for the repo's OllamaLLM adapter (src/llms/ollama.py):
# the same invoke / json_object contract the lab's extractor relies on.
TRIPLE_SCHEMA = '{"triples": [{"head": "...", "relation": "...", "tail": "..."}]}'


class _OllamaLLM:
    """Generate / extract with the locally served Ollama model.

    Contract mirror of the repo's OllamaLLM adapter, built inline so this
    notebook needs no repo imports: ``invoke`` returns the chat text,
    ``json_object`` strips a surrounding markdown code fence and re-prompts
    on parse failure.
    """

    def __init__(self, model: str = "qwen2.5-coder:7b", temperature: float = 0.0,
                 base_url: str = "http://localhost:11434"):
        self.model = model
        self.temperature = temperature
        self.base_url = base_url
        self._llm = None

    def _get_llm(self) -> ChatOllama:
        if self._llm is None:
            self._llm = ChatOllama(
                model=self.model, temperature=self.temperature, base_url=self.base_url
            )
        return self._llm

    def invoke(self, prompt: str) -> str:
        return self._get_llm().invoke(prompt).content

    @staticmethod
    def _strip_code_fence(text: str) -> str:
        """Remove a surrounding markdown code fence (```json ... ```)."""
        lines = text.strip().splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        return "\n".join(lines).strip()

    def json_object(self, prompt: str, retries: int = 2) -> dict:
        """Ask the model to output ONLY a JSON object and parse it."""
        text = ""
        for attempt in range(retries + 1):
            full = prompt if attempt == 0 else prompt + (
                "\n\nRespond with ONLY valid JSON, no markdown.")
            text = self.invoke(full)
            try:
                parsed = json.loads(self._strip_code_fence(text))
                if isinstance(parsed, (dict, list)):
                    return parsed
            except (json.JSONDecodeError, ValueError):
                pass
        return {"error": f"could not parse JSON after {retries + 1} attempts",
                "raw": text}


def extract_triples(llm, text: str) -> list[tuple[str, str, str]]:
    """Extract ``(head, relation, tail)`` triples from one passage of text."""
    prompt = (
        "Extract the entity-relation triples from the text below.\n"
        "Rules:\n"
        "- Only entities explicitly named in the text.\n"
        "- Entities are people, places, organizations, works, events or "
        "concrete things (1-4 words).\n"
        "- Relations are short verbs or prepositional phrases (1-4 words), "
        "present tense.\n"
        f"- Output ONLY JSON: {TRIPLE_SCHEMA}\n"
        "- Output an empty list if the text has no meaningful triples.\n"
        "\n"
        f"Text:\n{text}"
    )
    result = llm.json_object(prompt)
    if isinstance(result, list):  # bare array of triples, no wrapper key
        raw = result
    elif isinstance(result, dict) and "error" not in result:
        raw = result.get("triples", result.get("edges", result.get("data", [])))
        if isinstance(raw, dict):  # single triple given without a list wrapper
            raw = [raw]
    else:
        return []
    triples: list[tuple[str, str, str]] = []
    for item in raw or []:
        if not isinstance(item, dict):
            continue
        head = str(item.get("head", item.get("subject", ""))).strip()
        relation = str(item.get("relation", item.get("predicate", ""))).strip()
        tail = str(item.get("tail", item.get("object", ""))).strip()
        if head and tail:
            triples.append((head, relation or "related to", tail))
    return triples


def build_entity_graph(
    passages: list[str],
    llm,
    progress=None,
) -> nx.Graph:
    """Fold every passage's triples into one networkx entity graph.

    Node attributes: ``passages`` (indices of passages mentioning the
    entity). Edge attributes: ``relations`` (extracted phrases) and
    ``weight`` (how many distinct relation phrases were extracted).
    """
    graph = nx.Graph()
    total = len(passages)
    for i, text in enumerate(passages):
        for head, relation, tail in extract_triples(llm, text):
            if head == tail:
                continue  # self-loops carry no structure
            graph.add_edge(head, tail, relations=set())
            for node in (head, tail):
                graph.nodes[node].setdefault("passages", [])
            graph.nodes[head]["passages"].append(i)
            graph.nodes[tail]["passages"].append(i)
            graph[head][tail]["relations"].add(relation)
        if progress is not None:
            progress(i + 1, total)
    for _, _, data in graph.edges(data=True):
        data["weight"] = len(data["relations"])
    return graph


def graph_stats(graph: nx.Graph) -> dict:
    """Compact summary of graph structure, safe for graphs of any size."""
    degrees = [d for _, d in graph.degree()]
    components = list(nx.connected_components(graph))
    return {
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "density": round(nx.density(graph), 4),
        "avg_degree": round(sum(degrees) / len(degrees), 2) if degrees else 0.0,
        "max_degree": max(degrees) if degrees else 0,
        "connected_components": len(components),
        "largest_component": max((len(c) for c in components), default=0),
    }


def top_entities(graph: nx.Graph, k: int = 8) -> list[tuple[str, int]]:
    """The ``k`` highest-degree entities, as ``(name, degree)`` pairs."""
    ranked = sorted(graph.degree(), key=lambda pair: pair[1], reverse=True)
    return [(name, int(degree)) for name, degree in ranked[:k]]


def sample_relations(
    graph: nx.Graph, k: int = 8, seed: int = 42
) -> list[tuple[str, str, str]]:
    """``k`` example ``(head, relation, tail)`` triples from the graph."""
    rng = random.Random(seed)
    edges = list(graph.edges(data=True))
    rng.shuffle(edges)
    out: list[tuple[str, str, str]] = []
    for u, v, data in edges:
        for relation in sorted(data["relations"]):
            out.append((u, relation, v))
        if len(out) >= k:
            break
    return out[:k]


def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    llm = _OllamaLLM()  # local qwen2.5-coder:7b; expose json_object for extraction

    t0 = time.perf_counter()
    graph = build_entity_graph(
        passages,
        llm,
        progress=lambda done, total: print(
            f"  extracted {done}/{total} passages", end="\r", flush=True
        ),
    )
    build_s = time.perf_counter() - t0

    return {
        "passages": passages,
        "graph": graph,
        "build_s": build_s,
        "stats": graph_stats(graph),
        "top": top_entities(graph, TOP_K),
        "examples": sample_relations(graph, N_SAMPLE_RELATIONS),
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the graph's shape (density, average/max degree, connected components), the highest-degree hubs, a handful of extracted triples, and the takeaway. The hubs are the interesting part — a high-degree entity is a *bridge*: the node a multi-hop question has to walk through, which is exactly the structure dense retrieval cannot see.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-01 — Entity/relation graph from passages")
    print(f"{len(exp['passages'])} passages, built in {exp['build_s']:.1f}s")
    print("=" * 66)

    s = exp["stats"]
    print(f"\n[1] Graph structure over {s['nodes']} entities / {s['edges']} "
          f"edges:")
    print(f"    density      : {s['density']}")
    print(f"    avg degree   : {s['avg_degree']}")
    print(f"    max degree   : {s['max_degree']}")
    print(f"    components   : {s['connected_components']} "
          f"(largest {s['largest_component']})")

    print(f"\n[2] Highest-degree entities (hubs):")
    for name, degree in exp["top"]:
        print(f"    {degree:3d}  {name}")

    print(f"\n[3] Example extracted triples:")
    for head, relation, tail in exp["examples"]:
        print(f"    {head} -[{relation}]-> {tail}")

    print(f"\n[4] Takeaway")
    print("    The graph is a lossy but *structured* summary of the corpus:")
    print("    entities become nodes and their co-occurring relations become")
    print("    edges. Hubs (high degree) are the entities that connect many")
    print("    others — the bridges a multi-hop question must walk across.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks — the same gate `python src/curriculum/08-graphrag/01-entity-graph.py --verify` runs: every edge endpoint is a real node, every node remembers its passages, minimum size (≥ 15 entities, ≥ 10 edges), no fully isolated entity graph, and at least one edge carrying a real relation phrase. The gate turns "the lab ran" into "the lab ran *correctly*"; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    s = exp["stats"]
    graph = exp["graph"]

    checks.append(("every edge endpoint is a node",
                   all(graph.has_node(u) and graph.has_node(v)
                       for u, v in graph.edges())))
    checks.append(("every node remembers its passages",
                   all(graph.nodes[n].get("passages") for n in graph.nodes())))
    checks.append((f"graph has >= 15 entities (got {s['nodes']})",
                   s["nodes"] >= 15))
    checks.append((f"graph has >= 10 edges (got {s['edges']})",
                   s["edges"] >= 10))
    checks.append(("avg degree >= 1.0 (no fully isolated entity graph)",
                   s["avg_degree"] >= 1.0))
    checks.append(("at least one edge carries a relation phrase",
                   any(graph[u][v].get("relations") for u, v in graph.edges())))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

25 passages means 25 local LLM extraction calls — expect a few minutes depending on your hardware, with the progress line counting up as the graph builder goes. No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The graph, its hubs, and a sample of the triples — the structured summary the corpus was compressed into.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, the extraction pipeline misbehaved: check that Ollama is serving `qwen2.5-coder:7b` and that the corpus parquet is intact.


In [ ]:
verify_gate(exp)
